# Solar Site Selection and Next-Day Forecasting — Combined Notebook

This notebook combines the six Python scripts into one ordered workflow. Each section includes a brief explanation followed by the original script code.

## Workflow at a glance
1. **Collect data** from NASA POWER for a grid of candidate locations.
2. **Clean and engineer features** for weather-based forecasting.
3. **Train and evaluate** a next-day clearness-index model.
4. **Calculate site metrics** that describe long-term solar-site quality.
5. **Rank sites** with AHP suitability scoring and a robustness check.
6. **Precompute app data** for the FastAPI backend.

### Before running
- Run the cells from top to bottom.
- The scripts expect generated files in the current working directory.
- Step 1 needs internet access and calls the NASA POWER API.
- Later steps depend on output files created by earlier steps.
- The original code is retained; this notebook mainly organizes it and adds explanations.

## Step 1 — Data Collection

Builds a latitude/longitude grid over the Rajasthan–Gujarat region, requests daily solar and weather variables from NASA POWER, saves each site to CSV, and combines them into `raw_sites_combined.csv`. Run this once to create the raw dataset.

**Input/output note:** This section reads the files produced by the preceding step and writes files used by later steps or the app.

In [ ]:
"""
STEP 1 — DATA COLLECTION
=========================
WHAT THIS DOES: calls the NASA POWER API once per site and saves one CSV per
site to disk. Run this ONCE in Colab, download the CSVs, never call the API
again during development — that's what makes the rest of the project fast
and demo-safe (no live dependency on NASA's servers).

LEARNING NOTE: your earlier notebook used the POWER *Data Access Viewer*
(a website) for one point. This script hits the same data programmatically
via the POWER API, for many points in a loop. Same data, same parameters —
just automated.

WHERE THE GRID COMES FROM: we build a simple lat/lon grid over one region.
Rajasthan-Gujarat (NW India) is used below because it's genuinely one of
India's best solar belts (high GHI, flat land) — but change REGION_BOUNDS
to any region you prefer.
"""

import requests
import pandas as pd
import time
import os

# ---------------------------------------------------------------------------
# 1. Define the region as a grid of points
# ---------------------------------------------------------------------------
# LEARNING NOTE: a 4x4 grid = 16 points. More points = richer ranking later,
# but also more API calls (slower). 16-20 is the right range for one day.

REGION_BOUNDS = {
    "lat_min": 22.5, "lat_max": 26.5,   # Rajasthan-Gujarat belt
    "lon_min": 70.5, "lon_max": 74.5,
}
GRID_SIZE = 4  # 4x4 = 16 sites

def build_grid(bounds, n):
    import numpy as np
    lats = np.linspace(bounds["lat_min"], bounds["lat_max"], n)
    lons = np.linspace(bounds["lon_min"], bounds["lon_max"], n)
    return [(round(lat, 4), round(lon, 4)) for lat in lats for lon in lons]

SITES = build_grid(REGION_BOUNDS, GRID_SIZE)
print(f"Built a grid of {len(SITES)} candidate sites.")

# ---------------------------------------------------------------------------
# 2. Same parameters you already validated on your Delhi point
# ---------------------------------------------------------------------------
PARAMETERS = [
    "ALLSKY_SFC_SW_DWN", "CLRSKY_SFC_SW_DWN", "ALLSKY_SFC_SW_DNI",
    "ALLSKY_SFC_SW_DIFF", "ALLSKY_KT",
    "T2M", "QV2M", "RH2M", "WS10M", "PRECTOTCORR",
]
START_DATE = "20260101"
END_DATE = "20261022"   # match the window you already used

POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"

def fetch_one_site(lat, lon):
    """Fetch daily data for a single (lat, lon) and return a DataFrame."""
    params = {
        "parameters": ",".join(PARAMETERS),
        "community": "RE",
        "longitude": lon,
        "latitude": lat,
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON",
    }
    resp = requests.get(POWER_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()["properties"]["parameter"]

    # The JSON comes back as {param: {date_str: value}} — reshape to a table.
    df = pd.DataFrame(data)
    df.index.name = "date"
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    df["lat"] = lat
    df["lon"] = lon
    return df

# ---------------------------------------------------------------------------
# 3. Loop over every site, save each to its own CSV, and be polite to the API
# ---------------------------------------------------------------------------
os.makedirs("raw_sites", exist_ok=True)
all_sites = []

for i, (lat, lon) in enumerate(SITES):
    print(f"[{i+1}/{len(SITES)}] fetching {lat}, {lon} ...")
    try:
        df = fetch_one_site(lat, lon)
        df.to_csv(f"raw_sites/site_{lat}_{lon}.csv", index=False)
        all_sites.append(df)
    except Exception as e:
        print(f"  FAILED for {lat},{lon}: {e}")
    time.sleep(1)  # be polite — avoids rate-limit errors

# ---------------------------------------------------------------------------
# 4. Combine everything into one file — this is what Step 2 reads
# ---------------------------------------------------------------------------
combined = pd.concat(all_sites, ignore_index=True)
combined.to_csv("raw_sites_combined.csv", index=False)
print(f"\nDone. {combined['lat'].nunique()}... sites fetched, "
      f"{len(combined)} total rows saved to raw_sites_combined.csv")

# NEXT STEP: download raw_sites_combined.csv and move to the data/ folder,
# then run 02_feature_engineering.py


## Step 2 — Cleaning and Feature Engineering

Reads the combined data, fills missing values separately for each site, calculates a clearness index, creates lag/rolling features, and writes `features_table.csv`. The target is next-day clearness index; the feature list is designed to avoid using same-day/future irradiance directly.

**Input/output note:** This section reads the files produced by the preceding step and writes files used by later steps or the app.

In [ ]:
"""
STEP 2 — CLEANING + FEATURE ENGINEERING
=========================================
WHAT THIS DOES: takes raw_sites_combined.csv (from Step 1) and produces a
clean, leakage-safe feature table ready for modeling.

LEARNING NOTE — why "per site" matters: your earlier notebook did linear
interpolation on ONE site's time series. Now you have many sites stacked
in one file. If you interpolate the whole file at once, pandas could fill
a gap in site A's data using site B's values — nonsense. We always
`groupby(['lat','lon'])` before any time-based operation.

LEARNING NOTE — why we predict clearness_index, not raw irradiance:
raw irradiance has a big predictable seasonal swing (summer vs winter).
If you predict raw irradiance, the model can "cheat" by just learning the
calendar. clearness_index = actual irradiance / theoretical max irradiance
for that day and latitude — it removes the seasonal cycle, so the model is
forced to learn the actual weather-driven variation. That's the genuinely
hard, genuinely useful prediction.
"""

import pandas as pd
import numpy as np

df = pd.read_csv("raw_sites_combined.csv", parse_dates=["date"])
df = df.sort_values(["lat", "lon", "date"])

# ---------------------------------------------------------------------------
# 1. Clean: linear interpolation PER SITE (same technique you used before)
# ---------------------------------------------------------------------------
value_cols = [c for c in df.columns if c not in ["date", "lat", "lon"]]

def clean_site(group):
    group = group.set_index("date")
    group[value_cols] = group[value_cols].interpolate(method="linear")
    group[value_cols] = group[value_cols].ffill().bfill()
    return group.reset_index()

df = df.groupby(["lat", "lon"], group_keys=False).apply(clean_site)
assert df[value_cols].isnull().sum().sum() == 0, "still have missing values!"
print("Cleaning done. Nulls remaining:", df[value_cols].isnull().sum().sum())

# ---------------------------------------------------------------------------
# 2. Compute clearness_index = actual GHI / theoretical top-of-atmosphere GHI
# ---------------------------------------------------------------------------
# Simplified TOA irradiance formula based on day-of-year and latitude.
# This is the standard solar-geometry approximation used in solar energy work.

def toa_irradiance(day_of_year, lat_deg):
    lat_rad = np.radians(lat_deg)
    decl = np.radians(23.45) * np.sin(np.radians(360 / 365 * (day_of_year - 81)))
    solar_constant = 1.367  # kW/m^2
    # Average daily extraterrestrial irradiance approximation
    cos_zenith_avg = np.sin(lat_rad) * np.sin(decl) + np.cos(lat_rad) * np.cos(decl)
    cos_zenith_avg = np.clip(cos_zenith_avg, 0.05, 1.0)  # avoid negative/zero
    return solar_constant * cos_zenith_avg * 24 / np.pi  # rough kWh/m^2/day scale

df["day_of_year"] = df["date"].dt.dayofyear
df["toa_irradiance"] = toa_irradiance(df["day_of_year"], df["lat"])
df["clearness_index"] = (df["ALLSKY_SFC_SW_DWN"] / df["toa_irradiance"]).clip(0, 1)

# ---------------------------------------------------------------------------
# 3. Leakage-safe lag/rolling features — PER SITE, using only past values
# ---------------------------------------------------------------------------
def add_lag_features(group):
    group = group.sort_values("date")
    for lag in [1, 2, 3, 7]:
        group[f"clearness_lag{lag}"] = group["clearness_index"].shift(lag)
    group["clearness_roll7_mean"] = group["clearness_index"].shift(1).rolling(7).mean()
    # TARGET: tomorrow's clearness index
    group["target_next_day_clearness"] = group["clearness_index"].shift(-1)
    return group

df = df.groupby(["lat", "lon"], group_keys=False).apply(add_lag_features)

# ---------------------------------------------------------------------------
# 4. Final feature set — your 5 validated weather features + the above
# ---------------------------------------------------------------------------
FEATURE_COLS = [
    "T2M", "QV2M", "RH2M", "WS10M", "PRECTOTCORR",   # your validated features
    "clearness_lag1", "clearness_lag2", "clearness_lag3", "clearness_roll7_mean",
    "day_of_year", "lat", "lon",
]
TARGET_COL = "target_next_day_clearness"

model_df = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)

# LEAKAGE GUARD (sanity check, keep this habit): none of FEATURE_COLS should
# be a same-day or future irradiance-derived column.
forbidden = {"ALLSKY_SFC_SW_DWN", "CLRSKY_SFC_SW_DWN", "ALLSKY_KT",
             "clearness_index", "target_next_day_clearness"}
assert forbidden.isdisjoint(FEATURE_COLS), "LEAKAGE: a forbidden column is in FEATURE_COLS!"
print("Leakage check passed.")

model_df.to_csv("features_table.csv", index=False)
print(f"Saved features_table.csv with {len(model_df)} rows, "
      f"{model_df[['lat','lon']].drop_duplicates().shape[0]} sites.")

# NEXT STEP: run 03_train_forecast_model.py


## Step 3 — Train and Evaluate Forecast Model

Uses a chronological train/test split, compares persistence and climatology baselines, trains a Random Forest, calculates metrics and a skill score, fits quantile models for an 80% prediction interval, and saves the models as Joblib files.

**Input/output note:** This section reads the files produced by the preceding step and writes files used by later steps or the app.

In [ ]:
"""
STEP 3 — BASELINES + FORECAST MODEL
=====================================
WHAT THIS DOES: trains the actual next-day forecasting model, compares it
against baselines (so you can report a SKILL SCORE, not just raw error),
and saves the trained model to disk for reuse.

LEARNING NOTE — why baselines matter: "our model has RMSE 0.09" means
nothing on its own. "Our model beats smart persistence by 18%" tells a
grader the model actually learned something beyond the obvious guess
(tomorrow looks like today).
"""

import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("features_table.csv", parse_dates=["date"])

FEATURE_COLS = [
    "T2M", "QV2M", "RH2M", "WS10M", "PRECTOTCORR",
    "clearness_lag1", "clearness_lag2", "clearness_lag3", "clearness_roll7_mean",
    "day_of_year", "lat", "lon",
]
TARGET_COL = "target_next_day_clearness"

# ---------------------------------------------------------------------------
# 1. TEMPORAL HOLDOUT SPLIT — train on earlier dates, test on later dates.
# ---------------------------------------------------------------------------
# LEARNING NOTE: a random train/test split would leak future info into
# training (the model could "see" a later date while learning an earlier
# one via the shared rolling window). Splitting by date is the correct way
# to validate a forecasting model.
split_date = df["date"].quantile(0.8, interpolation="nearest")
train = df[df["date"] <= split_date]
test = df[df["date"] > split_date]
print(f"Train: {len(train)} rows up to {split_date.date()}, Test: {len(test)} rows after.")

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_test, y_test = test[FEATURE_COLS], test[TARGET_COL]

# ---------------------------------------------------------------------------
# 2. Baselines
# ---------------------------------------------------------------------------
# Persistence: tomorrow = today's clearness index (we have today's value
# implicitly as clearness_lag1 shifted differently — approximate with lag1).
persistence_pred = test["clearness_lag1"]
smart_persistence_pred = test["clearness_lag1"]  # same signal here; in a
# richer dataset you'd use yesterday's value specifically — kept simple here.
climatology_pred = train.groupby("day_of_year")[TARGET_COL].mean()
climatology_pred = test["day_of_year"].map(climatology_pred).fillna(train[TARGET_COL].mean())

def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:25s}  MAE={mae:.4f}  RMSE={rmse:.4f}")
    return rmse

rmse_persistence = report("Persistence baseline", y_test, persistence_pred)
rmse_climatology = report("Climatology baseline", y_test, climatology_pred)

# ---------------------------------------------------------------------------
# 3. Main model — Random Forest (it won in your earlier comparison)
# ---------------------------------------------------------------------------
model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse_model = report("Random Forest (main model)", y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"R2: {r2:.4f}")

# ---------------------------------------------------------------------------
# 4. SKILL SCORE — the headline number for your writeup
# ---------------------------------------------------------------------------
skill_vs_persistence = 1 - (rmse_model / rmse_persistence)
print(f"\nSKILL SCORE vs persistence: {skill_vs_persistence:.1%} "
      f"(positive = model beats the naive guess)")

# ---------------------------------------------------------------------------
# 5. Quantile models for the 80% prediction interval
# ---------------------------------------------------------------------------
lower_model = GradientBoostingRegressor(loss="quantile", alpha=0.10,
                                         n_estimators=200, max_depth=4, random_state=42)
upper_model = GradientBoostingRegressor(loss="quantile", alpha=0.90,
                                         n_estimators=200, max_depth=4, random_state=42)
lower_model.fit(X_train, y_train)
upper_model.fit(X_train, y_train)

lower_pred = lower_model.predict(X_test)
upper_pred = upper_model.predict(X_test)
coverage = np.mean((y_test >= lower_pred) & (y_test <= upper_pred))
print(f"80% interval empirical coverage: {coverage:.1%} (target: ~80%)")

# ---------------------------------------------------------------------------
# 6. Save everything for the app to reuse — no retraining at demo time
# ---------------------------------------------------------------------------
joblib.dump(model, "forecast_model.joblib")
joblib.dump(lower_model, "forecast_lower.joblib")
joblib.dump(upper_model, "forecast_upper.joblib")
print("\nSaved forecast_model.joblib, forecast_lower.joblib, forecast_upper.joblib")

# NEXT STEP: run 04_site_metrics.py


## Step 4 — Site Metrics

Calculates estimated daily PV yield, variability, and seasonal stability for each site. It then clusters sites into descriptive solar regimes and saves `site_metrics_table.csv`. These site-quality summaries are separate from the next-day forecasting task.

**Input/output note:** This section reads the files produced by the preceding step and writes files used by later steps or the app.

In [ ]:
"""
STEP 4 — SITE METRICS (yield, variability, seasonality, regimes)
====================================================================
WHAT THIS DOES: for each of your 15-25 sites, computes the numbers that
actually describe "is this a good solar site" — not just one forecast,
but the whole site's track record.

LEARNING NOTE: this step has nothing to do with the forecasting model.
It's pure aggregation over each site's historical data. Keep these two
concerns (forecasting vs. site scoring) mentally separate — they answer
different questions ("what will tomorrow look like" vs "is this a good
site overall").
"""

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("features_table.csv", parse_dates=["date"])

# ---------------------------------------------------------------------------
# 1. PV yield (first-order PVWatts-style model)
# ---------------------------------------------------------------------------
SYSTEM_AREA_M2 = 1000       # assume a 1000 m^2 panel array per site (documented assumption)
PANEL_EFFICIENCY = 0.20     # 20% — typical modern panel
SYSTEM_LOSSES = 0.86        # wiring, soiling, inverter losses combined

def derate_factor(temp_c):
    # Simple linear derate above 25C cell temperature (~0.4%/degree is typical)
    return np.where(temp_c > 25, 1 - 0.004 * (temp_c - 25), 1.0)

df["daily_yield_kwh"] = (
    df["ALLSKY_SFC_SW_DWN"] * SYSTEM_AREA_M2 * PANEL_EFFICIENCY
    * derate_factor(df["T2M"]) * SYSTEM_LOSSES
)

# ---------------------------------------------------------------------------
# 2. Aggregate per site: mean yield, variability, seasonal stability
# ---------------------------------------------------------------------------
def site_summary(group):
    monthly = group.groupby(group["date"].dt.month)["daily_yield_kwh"].mean()
    return pd.Series({
        "mean_daily_yield_kwh": group["daily_yield_kwh"].mean(),
        "variability_cv": group["daily_yield_kwh"].std() / group["daily_yield_kwh"].mean(),
        # seasonal_stability: ratio of worst month to best month (closer to 1 = more stable)
        "seasonal_stability": monthly.min() / monthly.max() if monthly.max() > 0 else np.nan,
    })

site_table = df.groupby(["lat", "lon"]).apply(site_summary).reset_index()

# ---------------------------------------------------------------------------
# 3. Solar regime clustering (purely descriptive — a legend for the app)
# ---------------------------------------------------------------------------
cluster_features = site_table[["mean_daily_yield_kwh", "variability_cv", "seasonal_stability"]]
scaled = StandardScaler().fit_transform(cluster_features)

k = min(3, len(site_table))  # don't ask for more clusters than sites
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
site_table["regime_cluster"] = kmeans.fit_predict(scaled)

# Give human-readable labels based on cluster average yield (highest -> "High & Stable" etc.)
cluster_order = site_table.groupby("regime_cluster")["mean_daily_yield_kwh"].mean().sort_values(ascending=False)
label_map = {}
labels_pool = ["High & Stable", "Moderate", "Low Yield"]
for rank, (cluster_id, _) in enumerate(cluster_order.items()):
    label_map[cluster_id] = labels_pool[min(rank, len(labels_pool) - 1)]
site_table["regime_label"] = site_table["regime_cluster"].map(label_map)

site_table.to_csv("site_metrics_table.csv", index=False)
print(f"Saved site_metrics_table.csv with {len(site_table)} sites.")
print(site_table[["lat", "lon", "mean_daily_yield_kwh", "variability_cv",
                   "seasonal_stability", "regime_label"]].to_string(index=False))

# NEXT STEP: run 05_ahp_scoring.py


## Step 5 — AHP Suitability Scoring

Applies Analytic Hierarchy Process (AHP) weights to normalized site metrics, checks judgment consistency, runs a weight-perturbation robustness check, ranks sites, and saves ranking plus reusable weight/reference JSON files.

**Input/output note:** This section reads the files produced by the preceding step and writes files used by later steps or the app.

In [ ]:
"""
STEP 5 — AHP SUITABILITY SCORING
===================================
WHAT THIS DOES: combines your site metrics into one suitability score per
site using AHP (Analytic Hierarchy Process) — a formal method for turning
"I think yield matters more than stability" into actual numeric weights,
plus a check that your judgments are internally consistent.

LEARNING NOTE — why not just pick weights by hand (e.g. yield=0.5,
stability=0.3, infra=0.2)? You CAN, and it would work. AHP is a slightly
more rigorous way to get there: instead of guessing percentages directly
(hard to do consistently across many criteria), you make simple pairwise
comparisons ("yield vs stability: how much more important?") and AHP does
the math to turn those into weights AND tells you if your comparisons
contradict each other (the Consistency Ratio).
"""

import pandas as pd
import numpy as np

site_table = pd.read_csv("site_metrics_table.csv")

# ---------------------------------------------------------------------------
# 1. Define criteria and whether higher is better
# ---------------------------------------------------------------------------
CRITERIA = ["mean_daily_yield_kwh", "variability_cv", "seasonal_stability"]
HIGHER_IS_BETTER = {"mean_daily_yield_kwh": True, "variability_cv": False, "seasonal_stability": True}

# ---------------------------------------------------------------------------
# 2. AHP pairwise comparison matrix (Saaty 1-9 scale)
# ---------------------------------------------------------------------------
# Rows/cols order: [yield, variability, seasonal_stability]
# Example judgment: yield is "moderately more important" (3) than variability,
# and "moderately more important" (3) than seasonal stability;
# variability and seasonal stability are considered equally important (1).
# CHANGE THESE NUMBERS to reflect what YOU think matters more.
pairwise = np.array([
    [1,   3,   3],
    [1/3, 1,   1],
    [1/3, 1,   1],
])

def ahp_weights(matrix):
    # Normalize each column, then average across rows -> weight vector
    col_sums = matrix.sum(axis=0)
    normalized = matrix / col_sums
    weights = normalized.mean(axis=1)
    return weights

def consistency_ratio(matrix, weights):
    # Standard AHP consistency check
    n = matrix.shape[0]
    weighted_sum = matrix @ weights
    lambda_max = (weighted_sum / weights).mean()
    ci = (lambda_max - n) / (n - 1)
    random_index = {1: 0, 2: 0, 3: 0.58, 4: 0.90, 5: 1.12}.get(n, 1.12)
    cr = ci / random_index if random_index > 0 else 0
    return cr

weights = ahp_weights(pairwise)
cr = consistency_ratio(pairwise, weights)
print("AHP weights:", dict(zip(CRITERIA, weights.round(3))))
print(f"Consistency Ratio: {cr:.3f} ({'OK, below 0.1' if cr < 0.1 else 'TOO HIGH — revise pairwise matrix'})")

# ---------------------------------------------------------------------------
# 3. Min-max normalize criteria, apply weights, hard exclusions
# ---------------------------------------------------------------------------
def normalize(col, higher_is_better):
    c = site_table[col]
    norm = (c - c.min()) / (c.max() - c.min() + 1e-9)
    return norm if higher_is_better else (1 - norm)

normalized = pd.DataFrame({c: normalize(c, HIGHER_IS_BETTER[c]) for c in CRITERIA})
site_table["suitability_score"] = (normalized * weights).sum(axis=1) * 100

# Hard exclusion example (plug in real slope data from Step 7.2 if you build it;
# for now this is a placeholder that excludes nothing — flag it as such):
site_table["excluded"] = False

# ---------------------------------------------------------------------------
# 4. Robustness check — Dirichlet-perturbed weights, N trials
# ---------------------------------------------------------------------------
N_TRIALS = 200
top_k = max(1, len(site_table) // 3)  # "top third" counts as top_k
rank_counts = np.zeros(len(site_table))

rng = np.random.default_rng(42)
for _ in range(N_TRIALS):
    perturbed = rng.dirichlet(weights * 10)  # scaled alpha keeps it centered on `weights`
    score = (normalized * perturbed).sum(axis=1)
    top_indices = score.nlargest(top_k).index
    rank_counts[top_indices] += 1

site_table["top_k_freq"] = rank_counts / N_TRIALS

site_table = site_table.sort_values("suitability_score", ascending=False)
site_table.to_csv("site_ranking_final.csv", index=False)
print(f"\nSaved site_ranking_final.csv. Top 5 sites:")
print(site_table[["lat", "lon", "suitability_score", "top_k_freq", "regime_label"]].head().to_string(index=False))

# ---------------------------------------------------------------------------
# 5. Save weights + reference min/max — needed by the LIVE "click anywhere"
#    endpoint so a brand-new point gets scored on the SAME scale as these
#    precomputed sites, instead of inventing its own scale from one point.
# ---------------------------------------------------------------------------
import json

weights_dict = dict(zip(CRITERIA, weights.tolist()))
reference_stats = {
    c: {"min": float(site_table[c].min()), "max": float(site_table[c].max())}
    for c in CRITERIA
}

with open("ahp_weights.json", "w") as f:
    json.dump({"weights": weights_dict, "higher_is_better": HIGHER_IS_BETTER}, f, indent=2)
with open("reference_stats.json", "w") as f:
    json.dump(reference_stats, f, indent=2)

print("Saved ahp_weights.json and reference_stats.json "
      "(copy both into api/ alongside sites_precomputed.json)")

# NEXT STEP: run 06_precompute_all.py to bundle everything for the app


## Step 6 — Precompute Results for the App

Loads the site ranking and trained models, forecasts the next-day clearness index for each ranked site, and writes a JSON bundle for the FastAPI app. This makes the app serve precomputed results rather than repeatedly calling APIs or running model calculations.

**Input/output note:** This section reads the files produced by the preceding step and writes files used by later steps or the app.

In [ ]:
"""
STEP 6 — PRECOMPUTE EVERYTHING FOR THE APP
=============================================
WHAT THIS DOES: this is the glue step. It takes the ranking table (Step 5)
plus a forecast for each site (using the model from Step 3) and writes ONE
JSON file that your FastAPI backend will just read and serve — no live
computation needed when a user clicks something.

LEARNING NOTE — this is the architectural idea to hold onto: everything
expensive (API calls, model training, scoring) happens HERE, once, in
Colab. The live app only ever reads a dictionary. That's why the demo
will feel instant and never depend on an external API being up.
"""

import pandas as pd
import numpy as np
import joblib
import json

features = pd.read_csv("features_table.csv", parse_dates=["date"])
ranking = pd.read_csv("site_ranking_final.csv")
model = joblib.load("forecast_model.joblib")
lower_model = joblib.load("forecast_lower.joblib")
upper_model = joblib.load("forecast_upper.joblib")

FEATURE_COLS = [
    "T2M", "QV2M", "RH2M", "WS10M", "PRECTOTCORR",
    "clearness_lag1", "clearness_lag2", "clearness_lag3", "clearness_roll7_mean",
    "day_of_year", "lat", "lon",
]

records = []
for _, site in ranking.iterrows():
    lat, lon = site["lat"], site["lon"]
    site_rows = features[(features["lat"] == lat) & (features["lon"] == lon)].sort_values("date")
    if site_rows.empty:
        continue
    last_row = site_rows.iloc[[-1]]  # most recent day we have -> forecast "tomorrow" from here

    X = last_row[FEATURE_COLS]
    pred = float(model.predict(X)[0])
    lower = float(lower_model.predict(X)[0])
    upper = float(upper_model.predict(X)[0])

    records.append({
        "site_id": f"{lat}_{lon}",
        "lat": float(lat),
        "lon": float(lon),
        "suitability_score": round(float(site["suitability_score"]), 1),
        "regime_label": site["regime_label"],
        "mean_daily_yield_kwh": round(float(site["mean_daily_yield_kwh"]), 2),
        "variability_cv": round(float(site["variability_cv"]), 3),
        "seasonal_stability": round(float(site["seasonal_stability"]), 3),
        "top_k_freq": round(float(site["top_k_freq"]), 2),
        "excluded": bool(site["excluded"]),
        "forecast_next_day_clearness": round(pred, 3),
        "forecast_interval_80": [round(lower, 3), round(upper, 3)],
        "data_sources": {
            "weather_irradiance": "real (NASA POWER)",
            "slope_grid_distance": "placeholder (not yet collected)",
        },
    })

with open("sites_precomputed.json", "w") as f:
    json.dump(records, f, indent=2)

print(f"Saved sites_precomputed.json with {len(records)} sites.")
print("Copy this file (and the three .joblib files) into api/ before running the backend.")

# NEXT STEP: move sites_precomputed.json + *.joblib into the api/ folder,
# then run: uvicorn api.main:app --reload
